In [ ]:
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
import numpy as np
import torch
from tqdm import tqdm

# Define the device (GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load the shapefile grid
grid = gpd.read_file("/home/stagiaire/Téléchargements/TE3/grille_finale.shp")

# Open the raster file
with rasterio.open("/home/stagiaire/Téléchargements/TE3/entropy_15x15.tif") as src:
    affine = src.transform
    raster_data = src.read(1)  # Read the first (and only) raster band
    raster_nodata = src.nodata

# Convert raster data to a PyTorch tensor and send it to the GPU
raster_tensor = torch.tensor(raster_data, dtype=torch.float32).to(device)

# Replace no-data values with NaN
if raster_nodata is not None:
    raster_tensor[raster_tensor == raster_nodata] = float('nan')

print(f"Raster converted to tensor with shape {raster_tensor.shape}")

# Function to compute the mean pixel value within a grid cell
def compute_cell_mean(geometry, affine, raster_tensor):
    """Calculate the mean pixel value inside a grid cell."""
    # Create a mask using the geometry of the grid cell
    mask = geometry_mask(
        [geometry], transform=affine, invert=True, out_shape=raster_tensor.shape
    ).astype(np.uint8)

    # Convert the mask to a PyTorch tensor and send it to the GPU
    mask_tensor = torch.tensor(mask, dtype=torch.bool).to(device)

    # Apply the mask to extract pixels within the cell
    cell_pixels = raster_tensor[mask_tensor]

    # Compute the mean of valid (non-NaN) pixels
    if cell_pixels.numel() > 0:
        mean_value = torch.nanmean(cell_pixels).item()  # Convert PyTorch result to Python scalar
    else:
        mean_value = np.nan

    return mean_value

# Function to save the grid as a shapefile
def save_grid(grid, index):
    """Save the grid with computed entropy values as a shapefile."""
    filename = f"grid_{index}.shp"
    print(f"Saving: {filename}")
    grid.to_file(filename)

# Initialize the new entropy column with NaN values
grid['entropy'] = np.nan

# Compute mean pixel values for each grid cell using the GPU
print("Computing mean pixel values for each grid cell...")

for idx, row in tqdm(grid.iterrows(), total=len(grid)):
    geometry = row['geometry']
    mean_value = compute_cell_mean(geometry, affine, raster_tensor)

    # Update the entropy column for the current cell
    grid.at[idx, 'entropy'] = mean_value

# Final save after processing all cells
save_grid(grid, "final")
print("Grid saved with computed entropy values!")
